# 05 — Training (MobileNetV3-Large)
Two-phase transfer learning with MobileNetV3-Large. OpenCV throughout, CUDA GPU.
6-class: Bacterial_Blight, Blast, Brown_Spot, Tungro, Healthy_Rice_Leaf, Hispa.
Companion model to ResNet-50 for comparative study.

## Imports & config

In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models
from torchvision.models import MobileNet_V3_Large_Weights, mobilenet_v3_large
from safetensors.torch import save_file, load_file
from pathlib import Path
from collections import Counter
import copy, time, warnings
warnings.filterwarnings('ignore')

# ── Config ────────────────────────────────────────────────────────────────────
PROCESSED_DIR = Path('../data/processed')
MODEL_PATH    = Path('../models/mobilenetv3_rice.safetensors')
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

CLASSES      = ['Bacterial_Blight', 'Blast', 'Brown_Spot', 'Tungro', 'Healthy_Rice_Leaf', 'Hispa']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
NUM_CLASSES  = len(CLASSES)

IMG_SIZE      = 224
BATCH_SIZE    = 32
NUM_WORKERS   = 4
MEAN          = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD           = np.array([0.229, 0.224, 0.225], dtype=np.float32)

PHASE1_EPOCHS = 10
PHASE1_LR     = 1e-3
PHASE2_EPOCHS = 20
PHASE2_LR     = 1e-4
ES_PATIENCE   = 5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {device}')
if device.type == 'cuda':
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'Classes : {CLASSES}')


## Augmentation pipeline (identical to ResNet-50 notebook)

In [ ]:
def random_horizontal_flip(img, p=0.5):
    return cv2.flip(img, 1) if np.random.rand() < p else img

def random_rotation(img, max_angle=15):
    angle = np.random.uniform(-max_angle, max_angle)
    h, w  = img.shape[:2]
    M     = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)

def random_color_jitter(img, brightness=0.2, contrast=0.2, hue_shift=10):
    alpha = 1.0 + np.random.uniform(-contrast, contrast)
    beta  = np.random.uniform(-brightness, brightness) * 255
    img   = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)
    if np.random.rand() < 0.4:
        hsv        = cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.int32)
        hsv[:,:,0] = (hsv[:,:,0] + np.random.randint(-hue_shift, hue_shift)) % 180
        img        = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
    return img

def random_crop(img, padding=20):
    h, w   = img.shape[:2]
    padded = cv2.copyMakeBorder(img, padding, padding, padding, padding,
                                cv2.BORDER_REFLECT_101)
    top    = np.random.randint(0, 2 * padding)
    left   = np.random.randint(0, 2 * padding)
    return padded[top:top+h, left:left+w]

def random_shadow(img, p=0.4):
    if np.random.rand() > p:
        return img
    h, w = img.shape[:2]
    pts  = np.array([
        [np.random.randint(0, w), 0],
        [np.random.randint(0, w), 0],
        [np.random.randint(0, w), h],
        [np.random.randint(0, w), h]
    ], np.int32)
    mask   = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [pts], 1)
    factor = np.random.uniform(0.4, 0.75)
    out    = img.copy().astype(np.float32)
    out[mask == 1] *= factor
    return out.clip(0, 255).astype(np.uint8)

def random_solid_bg_patch(img, p=0.3):
    if np.random.rand() > p:
        return img
    h, w = img.shape[:2]
    out  = img.copy()
    if np.random.rand() < 0.5:
        color = [np.random.randint(20, 80),
                 np.random.randint(50, 130),
                 np.random.randint(120, 200)]
    else:
        c = np.random.randint(30, 200); color = [c, c, c]
    side  = np.random.choice(['left', 'right', 'top', 'bottom'])
    cut_w = np.random.randint(w // 5, w // 2)
    cut_h = np.random.randint(h // 5, h // 2)
    if   side == 'left':   out[:, :cut_w]   = color
    elif side == 'right':  out[:, w-cut_w:] = color
    elif side == 'top':    out[:cut_h, :]   = color
    else:                  out[h-cut_h:, :] = color
    return out

def random_jpeg_compression(img, p=0.4, low=55, high=90):
    if np.random.rand() > p:
        return img
    bgr     = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    q       = np.random.randint(low, high)
    _, enc  = cv2.imencode('.jpg', bgr, [cv2.IMWRITE_JPEG_QUALITY, q])
    bgr_dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)
    return cv2.cvtColor(bgr_dec, cv2.COLOR_BGR2RGB)

def augment(img):
    img = random_horizontal_flip(img)
    img = random_rotation(img)
    img = random_color_jitter(img)
    img = random_crop(img)
    img = random_shadow(img)
    img = random_solid_bg_patch(img)
    img = random_jpeg_compression(img)
    return img

def preprocess(img):
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    img = img.astype(np.float32) / 255.0
    return (img - MEAN) / STD

def to_tensor(img):
    return torch.from_numpy(np.transpose(img, (2, 0, 1)).copy())

print('Augmentation pipeline ready.')


## Dataset & DataLoader

In [ ]:
class RiceLeafDataset(Dataset):
    def __init__(self, root, train=False):
        self.train   = train
        self.samples = []
        for cls in CLASSES:
            cls_dir = Path(root) / cls
            if not cls_dir.exists():
                print(f'  [WARNING] Missing folder: {cls_dir}')
                continue
            for ext in ('*.jpg', '*.jpeg', '*.JPG', '*.JPEG', '*.png'):
                for p in cls_dir.glob(ext):
                    self.samples.append((p, CLASS_TO_IDX[cls]))
        if not self.samples:
            raise RuntimeError(f'No images found under {root}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(str(path))
        if img is None:
            raise RuntimeError(f'Failed to read: {path}')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.train:
            img = augment(img)
        img = preprocess(img)
        return to_tensor(img), label


def make_weighted_sampler(dataset):
    counts     = Counter(label for _, label in dataset.samples)
    sample_wts = [1.0 / counts[label] for _, label in dataset.samples]
    return WeightedRandomSampler(sample_wts, num_samples=len(dataset), replacement=True)


train_ds = RiceLeafDataset(PROCESSED_DIR / 'train', train=True)
val_ds   = RiceLeafDataset(PROCESSED_DIR / 'val',   train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          sampler=make_weighted_sampler(train_ds),
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

train_counts = Counter(label for _, label in train_ds.samples)
val_counts   = Counter(label for _, label in val_ds.samples)
print(f'\n{"Class":<22} {"Train":>8} {"Val":>8}')
print('-' * 42)
for i, cls in enumerate(CLASSES):
    print(f'{cls:<22} {train_counts[i]:>8} {val_counts[i]:>8}')
print('-' * 42)
print(f'{"TOTAL":<22} {len(train_ds):>8} {len(val_ds):>8}')
print(f'\nTrain batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')


## Model — RiceMobileNet (MobileNetV3-Large)

Key differences from ResNet-50:
- `backbone.features` instead of `children()[:-1]`
- Explicit `AdaptiveAvgPool2d` — not built-in
- Feature size 960 instead of 2048
- Unfreeze last 4 blocks of `features` (index >= 13) in Phase 2
- ~5M params vs ~25M — 5x lighter, faster inference

In [ ]:
class RiceMobileNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, pretrained=True):
        super().__init__()
        weights       = MobileNet_V3_Large_Weights.IMAGENET1K_V1 if pretrained else None
        backbone      = mobilenet_v3_large(weights=weights)
        # Keep only the feature extractor blocks — strip the built-in classifier
        self.features = backbone.features          # 17 blocks (index 0–16)
        self.pool     = nn.AdaptiveAvgPool2d(1)    # must add manually — not built-in
        self.head     = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(960, 128),                   # 960 not 2048
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)   # (B, 960, 7, 7)
        x = self.pool(x)       # (B, 960, 1, 1)
        return self.head(x)

    def freeze_backbone(self):
        for p in self.features.parameters(): p.requires_grad = False
        for p in self.pool.parameters():     p.requires_grad = True
        for p in self.head.parameters():     p.requires_grad = True

    def unfreeze_top(self):
        # features has 17 blocks (0–16), unfreeze last 4 (index >= 13)
        for i, block in enumerate(self.features):
            for p in block.parameters():
                p.requires_grad = i >= 13

    def trainable(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


model = RiceMobileNet(pretrained=True).to(device)
total = sum(p.numel() for p in model.parameters())
print(f'Model   : MobileNetV3-Large  ({total/1e6:.1f}M total params)')
print(f'Classes : {NUM_CLASSES}')
print(f'Device  : {device}')


## Training utilities

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)


class EarlyStopping:
    def __init__(self, patience=5):
        self.patience   = patience
        self.counter    = 0
        self.best_loss  = float('inf')
        self.best_state = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss  = val_loss
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter    = 0
            return False
        self.counter += 1
        print(f'  EarlyStopping {self.counter}/{self.patience}')
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_state:
            model.load_state_dict(self.best_state)
            print(f'  Best weights restored (val_loss={self.best_loss:.4f})')


def run_epoch(model, loader, optimizer, train):
    model.train() if train else model.eval()
    total_loss = total_correct = total_n = 0
    with torch.set_grad_enabled(train):
        for imgs, labels in loader:
            imgs   = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = model(imgs)
            loss   = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss    += loss.item() * imgs.size(0)
            total_correct += (logits.argmax(1) == labels).sum().item()
            total_n       += imgs.size(0)
    return total_loss / total_n, total_correct / total_n


history = {
    'phase1': {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []},
    'phase2': {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []},
}

print('Training utilities ready.')


## Phase 1 — Head training (backbone frozen)

In [ ]:
print('=' * 55)
print('PHASE 1 — Head training (backbone frozen)')
print('=' * 55)
model.freeze_backbone()
print(f'Trainable params: {model.trainable():,}\n')

optimizer1 = Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=PHASE1_LR
)
scheduler1 = ReduceLROnPlateau(optimizer1, mode='min', factor=0.5, patience=3)

for epoch in range(1, PHASE1_EPOCHS + 1):
    t0              = time.time()
    tr_loss, tr_acc = run_epoch(model, train_loader, optimizer1, train=True)
    vl_loss, vl_acc = run_epoch(model, val_loader,   optimizer1, train=False)
    scheduler1.step(vl_loss)
    lr_now          = optimizer1.param_groups[0]['lr']
    print(f'Epoch {epoch:02d}/{PHASE1_EPOCHS} | '
          f'train_loss={tr_loss:.4f} acc={tr_acc:.4f} | '
          f'val_loss={vl_loss:.4f} acc={vl_acc:.4f} | '
          f'lr={lr_now:.2e} | {time.time()-t0:.1f}s')
    history['phase1']['train_loss'].append(tr_loss)
    history['phase1']['val_loss'].append(vl_loss)
    history['phase1']['train_acc'].append(tr_acc)
    history['phase1']['val_acc'].append(vl_acc)

print('\nPhase 1 complete.')


## Phase 2 — Fine-tuning (last 4 feature blocks unfrozen)

In [ ]:
print('=' * 55)
print('PHASE 2 — Fine-tuning (features[13–16] unfrozen)')
print('=' * 55)
model.unfreeze_top()
print(f'Trainable params: {model.trainable():,}\n')

optimizer2 = Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=PHASE2_LR
)
scheduler2 = ReduceLROnPlateau(optimizer2, mode='min', factor=0.5, patience=3)
stopper    = EarlyStopping(patience=ES_PATIENCE)

for epoch in range(1, PHASE2_EPOCHS + 1):
    t0              = time.time()
    tr_loss, tr_acc = run_epoch(model, train_loader, optimizer2, train=True)
    vl_loss, vl_acc = run_epoch(model, val_loader,   optimizer2, train=False)
    scheduler2.step(vl_loss)
    lr_now          = optimizer2.param_groups[0]['lr']
    print(f'Epoch {epoch:02d}/{PHASE2_EPOCHS} | '
          f'train_loss={tr_loss:.4f} acc={tr_acc:.4f} | '
          f'val_loss={vl_loss:.4f} acc={vl_acc:.4f} | '
          f'lr={lr_now:.2e} | {time.time()-t0:.1f}s')
    history['phase2']['train_loss'].append(tr_loss)
    history['phase2']['val_loss'].append(vl_loss)
    history['phase2']['train_acc'].append(tr_acc)
    history['phase2']['val_acc'].append(vl_acc)
    if stopper.step(vl_loss, model):
        print(f'Early stopping at epoch {epoch}')
        break

stopper.restore(model)
print('\nPhase 2 complete.')


## Save model

In [ ]:
save_file(model.state_dict(), str(MODEL_PATH))
print(f'Model saved  → {MODEL_PATH}')
print(f'Classes      : {CLASSES}')
print(f'Best val_loss: {stopper.best_loss:.4f}')


## Training curves

In [ ]:
import matplotlib.pyplot as plt

train_loss = history['phase1']['train_loss'] + history['phase2']['train_loss']
val_loss   = history['phase1']['val_loss']   + history['phase2']['val_loss']
train_acc  = history['phase1']['train_acc']  + history['phase2']['train_acc']
val_acc    = history['phase1']['val_acc']    + history['phase2']['val_acc']
p1_len     = len(history['phase1']['train_loss'])
epochs     = range(1, len(train_loss) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, train_loss, label='Train')
axes[0].plot(epochs, val_loss,   label='Val')
axes[0].axvline(p1_len, color='gray', linestyle='--', label='P1→P2')
axes[0].set_title('Loss — MobileNetV3')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs, train_acc, label='Train')
axes[1].plot(epochs, val_acc,   label='Val')
axes[1].axvline(p1_len, color='gray', linestyle='--', label='P1→P2')
axes[1].set_title('Accuracy — MobileNetV3')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/mobilenetv3_training_curves.png', dpi=150)
plt.show()
print(f'Final val acc  : {val_acc[-1]:.4f}')
print(f'Final val loss : {val_loss[-1]:.4f}')


## Load model utility

In [ ]:
def load_model(path=MODEL_PATH):
    m = RiceMobileNet(pretrained=False).to(device)
    m.load_state_dict(load_file(str(path)))
    m.eval()
    print(f'Model loaded from {path}')
    return m

# Uncomment to reload without retraining:
# model = load_model()
